In [19]:
from collections import deque
from typing import Optional, List, Any


class TreeNode:
    def __init__(self, val: Any = 0,
                 left: Optional["TreeNode"] = None,
                 right: Optional["TreeNode"] = None):
        self.val = val
        self.left = left
        self.right = right
        
def print_tree(root):
    """
    Печатает дерево сверху вниз по уровням.
    """
    if root is None:
        print("<empty tree>")
        return

    # Собираем все уровни через BFS
    levels = []
    current = [root]
    while any(n is not None for n in current):
        levels.append(current)
        nxt = []
        for n in current:
            if n is None:
                nxt.extend([None, None])
            else:
                nxt.extend([n.left, n.right])
        current = nxt

    h = len(levels)
    width = 2 ** h - 1  # ширина сетки для полного дерева высотой h

    for l, level in enumerate(levels):
        line = [" "] * width
        for i, node in enumerate(level):
            if node is None:
                continue
            # позиция узла в строке: по центру его «поддерева»
            col = (2 * i + 1) * 2 ** (h - l - 1) - 1
            s = str(node.val)
            start = col - (len(s) - 1) // 2
            for j, ch in enumerate(s):
                if 0 <= start + j < width:
                    line[start + j] = ch
        print("".join(line).rstrip())

1. **Восстановление дерева из массива.**  
   Корень — `arr[0]`. Для узла с индексом `i` дети имеют индексы `2*i + 1` и `2*i + 2`. Чтобы корректно обрабатывать `None`-пропуски, используем очередь `deque`: берём текущего родителя из очереди и по очереди назначаем ему левого и правого ребёнка, добавляя непустых детей в очередь. Сложность — `O(n)`.

In [20]:
def build_tree(arr):
    if not arr or arr[0] is None:
        return None

    root = TreeNode(arr[0])
    queue = deque([root])
    i = 1
    n = len(arr)

    while queue and i < n:
        node = queue.popleft()

        if i < n and arr[i] is not None:
            node.left = TreeNode(arr[i])
            queue.append(node.left)
        i += 1

        if i < n and arr[i] is not None:
            node.right = TreeNode(arr[i])
            queue.append(node.right)
        i += 1

    return root

tree = build_tree([1, 2, 3, None, 4])
print("Корень:", tree.val)
print_tree(tree)

Корень: 1
   1
 2   3
  4


2. **Симметричное дерево.**  
   Проверяем зеркальность двух поддеревьев. Сравниваем пары: `left.left` с `right.right` и `left.right` с `right.left`. Если оба узла `None` — пара симметрична; если только один `None` или значения различаются — нет. Рекурсивно проверяем все пары. Сложность — `O(n)`.

In [21]:
def is_symmetric(root):
    if root is None:
        return True

    def mirror(a, b):
        if a is None and b is None:
            return True
        if a is None or b is None:
            return False
        return (
            a.val == b.val
            and mirror(a.left, b.right)
            and mirror(a.right, b.left)
        )

    return mirror(root.left, root.right)

symmetric_tree = build_tree([1, 2, 2, 3, 4, 4, 3])
print("Симметрично:", is_symmetric(symmetric_tree))
print_tree(symmetric_tree)

Симметрично: True
   1
 2   2
3 4 4 3


3. **Минимальная глубина.**  
   Используем BFS. BFS обходит узлы по уровням, поэтому первый встреченный лист находится на минимальной глубине. Для каждого узла храним глубину; если у узла нет детей — сразу возвращаем глубину. Сложность — `O(n)`.

In [22]:
def min_depth(root):
    if root is None:
        return 0

    queue = deque([(root, 0)])  # корень на расстоянии 0 рёбер

    while queue:
        node, depth = queue.popleft()

        if node.left is None and node.right is None:
            return depth  # лист — возвращаем расстояние в рёбрах

        if node.left is not None:
            queue.append((node.left, depth + 1))
        if node.right is not None:
            queue.append((node.right, depth + 1))

    return 0

tree2 = build_tree([3, 9, 20, None, None, 1, 7])
print("Минимальная глубина:", min_depth(tree2))
print_tree(tree2)

Минимальная глубина: 1
   3
 9   20
    1 7


4. **Произведение min и max в BST.**  
   В BST минимальный элемент — самый левый узел, максимальный — самый правый. После восстановления дерева из level-order массива идём от корня влево до конца для минимума и вправо до конца для максимума. Это `O(h)`, где `h` — высота дерева, так как на каждом шаге спускаемся только в одну сторону.

In [23]:
def bst_min(root):
    if root is None:
        return None
    while root.left is not None:
        root = root.left
    return root.val


def bst_max(root):
    if root is None:
        return None
    while root.right is not None:
        root = root.right
    return root.val


def product_min_max_bst(arr):
    root = build_tree(arr)
    mn = bst_min(root)
    mx = bst_max(root)

    if mn is None or mx is None:
        return None
    return mn * mx

bst_arr = [4, 2, 6, 1, 3, 5, 7]
print("Произведение min и max BST:", product_min_max_bst(bst_arr))

Произведение min и max BST: 7


5. **Сравнение двух деревьев.**  
   Деревья идентичны, если: оба узла `None`; если один `None`, а другой нет — не идентичны; значения узлов равны и левые/правые поддеревья также идентичны. Рекурсивная проверка. Сложность — `O(n)`.

In [24]:
def is_same_tree(p, q):
    if p is None and q is None:
        return True
    if p is None or q is None:
        return False
    return (
        p.val == q.val
        and is_same_tree(p.left, q.left)
        and is_same_tree(p.right, q.right)
    )
    
p = build_tree([1, 2, 3])
q = build_tree([1, 2, 3])
print("Деревья идентичны:", is_same_tree(p, q))

p = build_tree([1, 2, 3])
q = build_tree([1, 2, None, 3])
print("Деревья идентичны:", is_same_tree(p, q))

Деревья идентичны: True
Деревья идентичны: False
